# Class 11: Cryptography with Python

## Learning outcomes

At the completion of this unit students should be able to:
1. Explain the key ideas of cryptography: plain text, cipher text, hashing, symmetric and asymmetric encryption
2. Encode strings to bytes, and decode bytes back to strings
3. Hash values with `hash()` and `hashlib`
4. Apply symmetric encryption with the `cryptography` module's `Fernet`
5. Apply asymmetric encryption with the `rsa` module

## 11.1 Terminology

- **Plain text**: the text which is readable, and which we want to keep as a secret.
- **Cipher text**: the message obtained after applying cryptography on plain text.
- **Hashing**: converting data into a unique string. The original data cannot be recovered from the hash string (it is a one-way process).
- **Encryption** (or encoding): the process of converting plain text to cipher text.
- **Decryption** (or decoding): the process of converting cipher text back to plain text.
- **Symmetric encryption**: when a plain text can be encrypted and decrypted using the *same* key.
- **DES (Data Encryption Standard)**: a symmetric block cipher standardized by NIST, limited to 56-bit keys. It is outdated, and superseded by AES because it is not secure.
- **AES (Advanced Encryption Standard)**: a symmetric block cipher standardized by NIST. AES is fast, cryptographically strong, and secure. It is a good default choice for encryption. For more information: https://en.wikipedia.org/wiki/Advanced_Encryption_Standard.
- **KDF (Key Derivation Function)**: a cryptographic hash function that derives one or more secret keys from a password. For more information: https://en.wikipedia.org/wiki/Key_derivation_function.
- **Fernet**: an implementation of symmetric cryptography. For more information: https://cryptography.io/en/latest/fernet/.

In this class we will focus on the Fernet class for symmetric cryptography.

## 11.2 Python's `r` and `b` string prefixes

A Python string literal can be prefixed by a letter that changes how Python interprets it. String prefixes are documented here: https://docs.python.org/3/reference/lexical_analysis.html#string-and-bytes-literals. We will briefly discuss two of them in today's class.

- A *raw* string literal is prefixed with `r`, like `r"Hi there!"`. Python takes its characters *literally* — no escape sequences are processed (handy for regular expressions and Windows paths). The result is still a normal `str`.
- A *byte* string literal is prefixed with `b` or `B`, like `b"Hi there!"`. It is an instance of the `bytes` type instead of the `str` type. In cryptography we deal with byte strings all the time, because encryption operates on *bytes*, not text.

Let's check the types using the `type()` function:

In [1]:
a = b"Hi there!"
print(type(a))

a = r"Hi there!"
print(type(a))

a = "Hi there!"
print(type(a))

<class 'bytes'>
<class 'str'>
<class 'str'>


In [2]:
b = b'Some text string'
print(b)
print('Decode byte to str: ',b.decode('utf-8'))

r = r'Some text string'
print(r)
print('Encode str to byte', r.encode('utf-8'))



b'Some text string'
Decode byte to str:  Some text string
Some text string
Encode str to byte b'Some text string'


Byte strings may only contain ASCII characters; bytes with a numeric value of 128 or greater must be expressed with escapes. To interconvert `str` and `bytes`, use `s.encode('utf-8')` (`str` -> `bytes`) and `b.decode('utf-8')` (`bytes` -> `str`), as shown above.

### Generation of random numbers and bytes

Random number and string generation is very important in cryptography, because it enables key generators to have unpredictable (random) features which cannot be predicted by adversaries. To generate a random number in Python, we use the `random()` function.

In [3]:
import random

random.randint(3,1000)

for i in range(10):
  print(random.random())

0.0012302628741114319
0.1520506272912141
0.3902420302981049
0.8399339922012409
0.6749457166146094
0.07383717320974126
0.8752866711753322
0.449490561879213
0.12626147610917005
0.7052838275155641


In [4]:
import numpy

numpy.random.seed(10)
print(numpy.random.random())

for _ in range(10):
    print(numpy.random.random())

0.771320643266746
0.0207519493594015
0.6336482349262754
0.7488038825386119
0.4985070123025904
0.22479664553084766
0.19806286475962398
0.7605307121989587
0.16911083656253545
0.08833981417401027
0.6853598183677972


These numbers are random numbers in the range from 0 to 1. We can also generate a random byte string using the `urandom()` function in the `os` module. Note, in the example below, that `urandom()` receives an integer parameter which is the size of the random byte string. The difference between these two is that `urandom()` does not depend on a seed, but it instead generates randomness from various sources. Hence, it can be considered more random than `random()`.


In [5]:
import os

a = os.urandom(1)
print(a)


b'\xa9'


## 11.3 Hashing

Hashing can be performed using the built-in function `hash()`. You can only hash immutable objects like numbers, strings and tuples.

In [6]:
print(hash('abC'))
print(len(str(hash('abC'))))

-9195328316239179873
20


Hashing mutable objects will give an error.

In [7]:
hash([1,2,3])

TypeError: unhashable type: 'list'

Another library to use for hashing is `hashlib`.

In [8]:
import hashlib

print(hashlib.md5(b'abc').hexdigest())

900150983cd24fb0d6963f7d28e17f72


You can make your own python object hashable by overriding the `__hash__()` class method.

In [9]:
class Employee:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    def __hash__(self):      
        return hash((self.first_name, self.last_name))

e = Employee('A','C')
print(hash(e))

2878341485291965251



## 11.4 Symmetric encryption: `Fernet`

The Fernet class, which is part of the `cryptography` package, offers symmetric encryption based on the AES. Fernet is a standard for encrypting buffers in a way that follows best-practices cryptography. It is not suitable for very big files.

In symmetric, or secret key, cryptography: the same key is used for encryption and decryption, and therefore must be kept safe.



In [10]:
# (if needed, install first:  pip install cryptography)
from cryptography.fernet import Fernet

key = Fernet.generate_key()
print('The key: ', key)
print('Creating a Fernet object with the key.')
fkey = Fernet(key)
cipher_text = fkey.encrypt(b"I am working from home.")
print("Cipher: ",cipher_text)
plain_text = fkey.decrypt(cipher_text)
print("Plain: ",plain_text)

The key:  b'nj3RvN0TIYMaLztXgX9R_v-5eywRmwmNpGbdTkzvLY4='
Creating a Fernet object with the key.
Cipher:  b'gAAAAABq2rJZrLHiTwLHBTcW674U5sf8lYzCLPazyIcxhd_J4rN2tMvBByWoc77NnZd4jGlod7qmKgc0k8dGzI-CSU50MIC-wT4YP3rduAv_pg30BeFNgJ4='
Plain:  b'I am working from home.'


You will get slightly different values if you encrypt on your machine.

Note that this only encrypts and decrypts byte strings. In order to encrypt and decrypt text strings, they will need to be encoded and decoded, usually with UTF-8.


In [11]:
from cryptography.fernet import Fernet

st = "I am working from home."
st=st.encode('utf-8')

key = Fernet.generate_key()
fkey = Fernet(key)
print('The key object: ', fkey)
cipher_text = fkey.encrypt(st)
print("Cipher: ",cipher_text)
plain_text = fkey.decrypt(cipher_text)
print("Plain: ",plain_text)

The key object:  <cryptography.fernet.Fernet object at 0x7fd36756ee90>
Cipher:  b'gAAAAABq2rJZwA7EVjMN9u81yuxHr67R4gBSgKzlf8PHTWrRL9fGXx8el0Iz3mfhfpxDS1Mw_S_JL3Oxv6IimXwm85RgDzYP6Eh9c5Z7Z3UMgoedsHenFLg='
Plain:  b'I am working from home.'


### Fernet key from a password

You can create a new key based on a password. There are many different `cryptography` classes that can *derive* a key based on a password. Here will use one of them, `PBKDF2HMAC`.

In [12]:
import os
import base64
from cryptography.fernet import Fernet
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
password = b"my complicated password"

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=os.urandom(10),
    iterations=100000,
    backend=default_backend()
)
pre_key = kdf.derive(password)
print(pre_key)

key = base64.urlsafe_b64encode(pre_key)
print(key)
fkey = Fernet(key)
print('The key object: ', fkey)
cipher_text = fkey.encrypt(st)
print("Cipher: ",cipher_text)
plain_text = fkey.decrypt(cipher_text)
print("Plain: ",plain_text)

b'\xb1=.\xc9\xa2o\xc4\nc\xf0v0Q\xaeB\xbb\x0ekR\xe3\xe3\xde\xc6\xb4\x0e\x8f\xe7\xe6\xb6\xfc\x82\xd5'
b'sT0uyaJvxApj8HYwUa5Cuw5rUuPj3sa0Do_n5rb8gtU='
The key object:  <cryptography.fernet.Fernet object at 0x7fd36756c2e0>
Cipher:  b'gAAAAABq2rJZ8Pxvplr1YFXTspWZm93T4hFe0mlGDPdz4Sy1zRUgt41h75id5O4SsgnD5Tp5sP9mrapUfrZAX80cEFHtylIfYCEgRjfWW1nxD3tQt_oEn7c='
Plain:  b'I am working from home.'


## 11.5 Asymmetric encryption

The `rsa` library is an implementation of asymmetric encryption. Here is an example:

In [13]:
import rsa

public_key, private_key = rsa.newkeys(300,accurate=False)
print(public_key)

message = b"This is a trade secret."

crypt = rsa.encrypt(message, public_key)

print(crypt)

decrypt = rsa.decrypt(crypt, private_key)

print(decrypt.decode())

PublicKey(797957571166023278370013801244286514262844354741750868763184881629884808118784668498588779, 65537)
b'\x03x\xb1W\x8cEb\xc3\x93\t{\xe9o\xb3M\xdd\xff\xfc\xbf\x89\x96i\xba9\xc5_\xb8E\x01\xce\xf9\xc5\xd3v\x95)D!'
This is a trade secret.


`nbits` determines the cryptographic strength of the key and the size of the message string that can be encrypted in one go. (In the real world you'd use 2048 bits or more; a key that small takes a little while to generate.)

Note the workflow, which is exactly backwards from `Fernet`: two *different* keys are generated — a **public key** that can be shared with the world, and a **private key** that must be kept secret. What is encrypted with the public key can only be decrypted with the matching private key. That's *asymmetric* encryption.

**GOTO Lab exercises 1-6**

## Summary

- Plain text + a key -> cipher text (encryption); cipher text + key -> plain text (decryption). A *hash* is a one-way fingerprint that can't be reversed.
- Strings prefixed with `b` are byte strings (`bytes`, not `str`); those with `r` are raw strings where backslashes aren't special.
- Convert text to bytes with `.encode('utf-8')`, and back with `.decode('utf-8')`.
- `random.random()` is fine for games, `os.urandom()` for cryptographic randomness.
- Hash with `hashlib` (e.g. `hashlib.md5(...)` or, better, `sha256`); only *immutable* values are hashable.
- `Fernet` (from the `cryptography` package) does *symmetric* encryption: one secret key encrypts and decrypts. A key can be *derived* from a password with a KDF such as `PBKDF2HMAC`.
- `rsa` does *asymmetric* encryption: a public key encrypts, the matching private key decrypts.